# Mission 3 ver4 — 저장된 최적 모델로 바로 예측

**Colab에서 GPU를 선택한 뒤 `런타임 → 모두 실행`을 누르세요.** Drive 연결 창이 나오면 연결합니다.
기존 폴더 구조라면 아래 경로를 수정하지 않아도 Validation 예측과 평가까지 실행됩니다.

- 다섯 모델의 최적 epoch **3 / 2 / 3 / 2 / 2**는 이미 `fold_1.ckpt`~`fold_5.ckpt`의 가중치에 반영되어 있습니다.
- 저장된 다섯 모델을 한 번씩 불러와 **예측을 평균**하고, 기존 TF-IDF 모델과 증상별 가중치·임계값을 적용합니다.
- **학습, epoch 반복, 파라미터 탐색, 보정값 재탐색을 실행하지 않습니다.** 기존 ver4 앙상블을 사용하므로 다섯 모델의 추론은 모두 필요합니다.
- 예측 CSV를 먼저 저장한 뒤 정답이 있는 데이터만 평가합니다. 정답 없는 Test JSON도 예측할 수 있습니다.

노트북에는 실행 코드가 포함되어 있습니다. **가중치는 아래처럼 Drive에 별도로 있어야 합니다.**
`mission3_ver4_model.ckpt` 하나만 올리면 실행할 수 없습니다. 기존 결과 폴더 전체를 유지하는 것이 편합니다.

```text
MyDrive/DCC_AIdeas/results/mission3/ver4/
├── mission3_ver4_model.ckpt   # 모델 구성과 보정값
├── fold_1.ckpt ... fold_5.ckpt
├── sparse.joblib
├── tokenizer/
├── fold_1_done.json ... fold_5_done.json   # 선택 사항: epoch 기록 표시
└── submission/mission3_ver4_inference.ipynb
```

기본 데이터 위치는 `MyDrive/DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급`
또는 `MyDrive/DCC_AIdeas/DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급`입니다.
다른 데이터는 다음 셀의 `LABEL_DIR`만 바꾸세요. 압축 파일은 미리 풀어 두세요.


In [ ]:
# 1. 기본 폴더 구조라면 그대로 모두 실행하세요.
# 직접 지정할 때 예: MODEL_DIR = "/content/drive/MyDrive/DCC_AIdeas/results/mission3/ver4"
MODEL_DIR = None          # mission3_ver4_model.ckpt, fold_*.ckpt 등이 있는 폴더
LABEL_DIR = None          # 예측할 JSON 폴더. None이면 VL_서울_구급 자동 탐색
OUTPUT_DIR = None         # None이면 모델 폴더/inference_outputs/실행시각/에 새로 저장
RUN_EVALUATION = True     # 모든 JSON에 symptom 정답이 있을 때만 평가
INFERENCE_BATCH_SIZE = 8  # 기존 ver4의 추론 설정. GPU 메모리가 부족하면 4 또는 2
DEVICE = "auto"          # auto: GPU가 있으면 cuda, 없으면 cpu


## 2. 실행 환경 준비

저장 당시의 NumPy·scikit-learn·Transformers 버전을 사용합니다.
호환되는 기존 PyTorch는 유지합니다. 설치 후 추론은 새 Python 프로세스에서 실행하므로
이미 불러온 패키지 때문에 노트북 런타임을 다시 시작할 필요가 없습니다.
패키지 설치와 Drive 연결에는 인터넷이 필요하며, 이후 모델을 새로 내려받지 않습니다.

저장 모델의 패키지 버전을 맞추는 이유: [scikit-learn 모델 저장 문서](https://scikit-learn.org/stable/model_persistence.html).


In [ ]:
import subprocess
import sys

REQUIREMENTS = [
    "torch>=2.6,<3", "transformers==4.57.1", "numpy==2.1.3",
    "scikit-learn==1.6.1", "scipy>=1.11,<2",
    "iterative-stratification==0.1.9", "joblib>=1.3,<2",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q", *REQUIREMENTS],
    check=True,
)
print("실행 패키지 준비 완료")


In [ ]:
# 3. Drive 연결 및 입력/출력 경로 확인
import json
import os
import tempfile
import uuid
from datetime import datetime, timedelta, timezone
from pathlib import Path

START_DIR = Path.cwd().resolve()
try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount("/content/drive", force_remount=False)

roots = ([Path("/content"), Path("/content/drive/MyDrive")] if IN_COLAB else [])
roots += [START_DIR, *list(START_DIR.parents)[:6]]
roots = list(dict.fromkeys(roots))

def choose_path(explicit, candidates, is_valid, setting, description):
    if explicit is not None:
        path = Path(explicit).expanduser().resolve()
        if not is_valid(path):
            raise FileNotFoundError(f"{setting}: {description}을 확인하세요: {path}")
        return path
    for candidate in dict.fromkeys(candidates):
        if is_valid(candidate):
            return candidate.resolve()
    raise FileNotFoundError(
        f"{description}을 찾지 못했습니다. 1번 셀의 {setting}에 실제 폴더 경로를 지정하세요."
    )

model_candidates = []
for root in roots:
    model_candidates.extend([
        root, root / "results/mission3/ver4", root / "DCC_AIdeas/results/mission3/ver4",
    ])
MODEL_DIR = choose_path(
    MODEL_DIR, model_candidates,
    lambda p: (p / "mission3_ver4_model.ckpt").is_file(),
    "MODEL_DIR", "저장된 ver4 모델 폴더",
)

label_candidates = []
for root in [*roots, *list(MODEL_DIR.parents)[:4]]:
    label_candidates.extend([root / "VL_서울_구급", root / "mission3_data/validation/VL_서울_구급"])
    for base in [root, root / "DATA", root / "data", root / "DCC_AIdeas/DATA",
                 root / "DCC_AIdeas/data", root / "mission3_data"]:
        for relative in ["대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급",
                         "Validation/2.라벨링데이터/VL_서울_구급"]:
            label_candidates.append(base / relative)
LABEL_DIR = choose_path(
    LABEL_DIR, label_candidates, lambda p: p.is_dir() and next(p.rglob("*.json"), None) is not None,
    "LABEL_DIR", "예측할 JSON 폴더",
)
if OUTPUT_DIR is None:
    stamp = datetime.now(timezone(timedelta(hours=9))).strftime("%Y%m%d_%H%M%S")
    OUTPUT_DIR = MODEL_DIR / "inference_outputs" / (stamp + "_" + uuid.uuid4().hex[:6])
OUTPUT_DIR = Path(OUTPUT_DIR).expanduser().resolve()
output_names = ["mission3.csv", "validation_metrics.json", "inference_summary.json"]
if any((OUTPUT_DIR / name).exists() for name in output_names):
    raise FileExistsError("기존 결과 보호: OUTPUT_DIR를 새 폴더로 지정하거나 None으로 바꾸세요.")
if not isinstance(INFERENCE_BATCH_SIZE, int) or isinstance(INFERENCE_BATCH_SIZE, bool) or INFERENCE_BATCH_SIZE < 1:
    raise ValueError("INFERENCE_BATCH_SIZE는 1 이상의 정수여야 합니다.")
if DEVICE not in {"auto", "cuda", "cpu"}:
    raise ValueError('DEVICE는 "auto", "cuda", "cpu" 중 하나여야 합니다.')

checkpoint_path = MODEL_DIR / "mission3_ver4_model.ckpt"
prediction_path = OUTPUT_DIR / "mission3.csv"
print("모델:", MODEL_DIR)
print("입력:", LABEL_DIR)
print("새 결과:", OUTPUT_DIR)


## 4. 포함된 실행 코드 준비

다음 셀은 기존 ver4 추론 코드를 임시 폴더에 준비합니다. 수정할 필요가 없습니다.
모델 폴더 안의 가중치·보정값은 읽기만 하며, 새 결과는 앞서 표시한 폴더에 저장합니다.


In [ ]:
# @title 포함된 ver4 실행 코드 (수정 없이 실행)
RUNTIME_SOURCES = {'data.py': '"""Mission 3: 모델 입력은 양측 발화의 text만, symptom은 학습/평가 정답만."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport unicodedata\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\nfrom iterstrat.ml_stratifiers import MultilabelStratifiedKFold, MultilabelStratifiedShuffleSplit\nfrom sklearn.metrics import f1_score, precision_recall_fscore_support\n\nTARGET_SYMPTOMS = (\n    "고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란",\n)\n\n\ndef dialogue_text(annotation: dict) -> str:\n    """입력/정답과 무관하게 동일하게 전처리하며 발화 순서를 유지한다."""\n    utterances = annotation.get("utterances")\n    if not isinstance(utterances, list):\n        utterances = annotation.get("utterences", [])\n    if not isinstance(utterances, list):\n        raise ValueError("utterances/utterences must be a list")\n    texts = []\n    for utterance in utterances:\n        if isinstance(utterance, dict) and isinstance(utterance.get("text"), str):\n            text = " ".join(unicodedata.normalize("NFC", utterance["text"]).split())\n            if text:\n                texts.append(text)\n    return "\\n".join(texts)\n\n\ndef target_vector(annotation: dict) -> list[int]:\n    if "symptom" not in annotation:\n        raise ValueError("Training/evaluation JSON has no symptom target")\n    raw = annotation["symptom"]\n    if raw is None:\n        raw = []\n    if isinstance(raw, str):\n        raw = [raw]\n    if not isinstance(raw, list) or any(not isinstance(x, str) for x in raw):\n        raise ValueError("symptom must be a string or a list of strings")\n    found = {x.strip() for x in raw}\n    return [int(name in found) for name in TARGET_SYMPTOMS]\n\n\ndef text_group(text: str) -> str:\n    # 파일명, 날짜, speaker 등은 모델 특징에도 중복 판단에도 사용하지 않는다.\n    normalized = " ".join(unicodedata.normalize("NFC", text).split())\n    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()\n\n\ndef load_dataset(label_dir: Path, include_targets: bool = False, workers: int = 8):\n    paths = sorted(Path(label_dir).rglob("*.json"))\n    if not paths:\n        raise ValueError(f"JSON 파일이 없습니다: {label_dir}")\n    if len({p.name for p in paths}) != len(paths):\n        raise ValueError("동일한 JSON 파일명이 중복됩니다. 제출 CSV의 식별자가 모호합니다.")\n\n    def read(path):\n        with path.open(encoding="utf-8-sig") as handle:\n            annotation = json.load(handle)\n        if not isinstance(annotation, dict):\n            raise ValueError(f"JSON object가 아닙니다: {path}")\n        # 추론 경로에서는 target_vector를 호출하거나 symptom에 접근하지 않는다.\n        return dialogue_text(annotation), target_vector(annotation) if include_targets else None\n\n    with ThreadPoolExecutor(max_workers=workers) as executor:\n        rows = list(executor.map(read, paths))\n    texts = [row[0] for row in rows]\n    targets = np.asarray([row[1] for row in rows], dtype=np.int8) if include_targets else None\n    return paths, texts, targets\n\n\ndef grouped_targets(texts, targets):\n    groups = np.asarray([text_group(text) for text in texts])\n    unique, inverse = np.unique(groups, return_inverse=True)\n    aggregate = np.zeros((len(unique), len(TARGET_SYMPTOMS)), dtype=np.int8)\n    np.maximum.at(aggregate, inverse, targets)\n    return groups, inverse, aggregate\n\n\ndef make_splits(texts, targets, calibration_size: float, n_folds: int, seed: int):\n    """전체 Training의 보정 집합은 모든 모델 fit/epoch 선택에서 분리한다."""\n    if not 0 < calibration_size < 0.5 or n_folds < 2:\n        raise ValueError("calibration_size must be in (0, .5); n_folds >= 2")\n    groups, inverse, aggregate = grouped_targets(texts, targets)\n    group_ids = np.arange(len(aggregate))\n    outer = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=calibration_size, random_state=seed)\n    dev_groups, cal_groups = next(outer.split(group_ids, aggregate))\n    development = np.flatnonzero(np.isin(inverse, dev_groups))\n    calibration = np.flatnonzero(np.isin(inverse, cal_groups))\n    splitter = MultilabelStratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed + 1)\n    folds = []\n    for train_g, stop_g in splitter.split(dev_groups, aggregate[dev_groups]):\n        train_indices = np.flatnonzero(np.isin(inverse, dev_groups[train_g]))\n        stop_indices = np.flatnonzero(np.isin(inverse, dev_groups[stop_g]))\n        folds.append((train_indices, stop_indices))\n    for name, indices in [("calibration", calibration), ("development", development),\n                          *[(f"fold{i}/{part}", ids) for i, pair in enumerate(folds)\n                            for part, ids in zip(("fit", "stop"), pair)]]:\n        counts = targets[indices].sum(axis=0)\n        if len(indices) == 0 or np.any(counts == 0) or np.any(counts == len(indices)):\n            raise ValueError(f"{name}: 각 증상의 양성/음성이 필요합니다. 데이터/분할 수를 확인하세요.")\n    return development, calibration, folds, groups\n\n\ndef write_json(path: Path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef write_predictions(paths, probabilities, thresholds, output: Path):\n    probabilities = np.asarray(probabilities)\n    if probabilities.shape != (len(paths), len(TARGET_SYMPTOMS)) or not np.isfinite(probabilities).all():\n        raise ValueError("Invalid prediction shape or non-finite probabilities")\n    predicted = probabilities >= np.asarray(thresholds)\n    output = Path(output)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with output.open("w", encoding="utf-8-sig", newline="") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["label file name", "symptom"])\n        for path, flags in zip(paths, predicted):\n            # 기존 제출 CSV와 같은 Python string-list 표현. 빈 list도 허용한다.\n            writer.writerow([Path(path).name, str([s for s, flag in zip(TARGET_SYMPTOMS, flags) if flag])])\n    return predicted.astype(np.int8)\n\n\ndef score_predictions(targets, predictions):\n    precision, recall, f1, support = precision_recall_fscore_support(targets, predictions, zero_division=0)\n    return {\n        "macro_f1": float(f1_score(targets, predictions, average="macro", zero_division=0)),\n        "samples": int(len(targets)),\n        "per_symptom": {name: {"f1": float(f1[i]), "precision": float(precision[i]),\n                                "recall": float(recall[i]), "support": int(support[i])}\n                        for i, name in enumerate(TARGET_SYMPTOMS)},\n    }\n', 'calibration.py': '"""Training-only grouped calibration for nine independent symptom labels."""\nfrom __future__ import annotations\n\nimport numpy as np\nfrom iterstrat.ml_stratifiers import MultilabelStratifiedKFold\nfrom sklearn.metrics import f1_score, precision_recall_fscore_support\n\nif __package__:\n    from .data import TARGET_SYMPTOMS\nelse:\n    from data import TARGET_SYMPTOMS\n\nWEIGHT_GRID = (0.0, 0.25, 0.5, 0.75, 1.0)\nALL_NEGATIVE_THRESHOLD = float(np.nextafter(1.0, np.inf))\n\n\ndef _probabilities(values, name):\n    values = np.asarray(values, dtype=np.float64)\n    if values.ndim != 2 or values.shape[1] != len(TARGET_SYMPTOMS):\n        raise ValueError(f"{name} must have shape (samples, {len(TARGET_SYMPTOMS)})")\n    if not len(values) or not np.isfinite(values).all():\n        raise ValueError(f"{name} must be nonempty and finite")\n    if np.any((values < 0) | (values > 1)):\n        raise ValueError(f"{name} must contain probabilities in [0, 1]")\n    return values\n\n\ndef _inputs(y_true, neural_probs, sparse_probs):\n    neural = _probabilities(neural_probs, "neural_probs")\n    sparse = _probabilities(sparse_probs, "sparse_probs")\n    truth = np.asarray(y_true)\n    if truth.shape != neural.shape or sparse.shape != neural.shape:\n        raise ValueError("truth, neural_probs and sparse_probs must have identical shapes")\n    if not np.isin(truth, (0, 1)).all():\n        raise ValueError("y_true must contain only binary labels")\n    return truth.astype(np.int8), neural, sparse\n\n\ndef _best_threshold(truth, probabilities):\n    """Exact F1 maximum for >= decisions; tied probabilities stay together."""\n    positives = int(truth.sum())\n    if positives == 0:\n        return ALL_NEGATIVE_THRESHOLD, 0.0\n    if positives == len(truth):\n        return 0.0, 1.0\n    order = np.argsort(-probabilities, kind="stable")\n    sorted_probs = probabilities[order]\n    # A threshold can include a complete tied block, never part of one.\n    boundaries = np.r_[np.flatnonzero(sorted_probs[:-1] != sorted_probs[1:]), len(truth) - 1]\n    true_positives = np.cumsum(truth[order])[boundaries]\n    scores = 2.0 * true_positives / (positives + boundaries + 1)\n    thresholds = np.zeros(len(boundaries), dtype=np.float64)\n    has_next = boundaries < len(truth) - 1\n    upper = sorted_probs[boundaries[has_next]]\n    lower = sorted_probs[boundaries[has_next] + 1]\n    midpoint = lower + (upper - lower) / 2\n    # Adjacent floating-point values may have no representable midpoint.\n    thresholds[has_next] = np.where(midpoint > lower, midpoint, upper)\n    best = np.flatnonzero(scores == scores.max())\n    # An explicit deterministic tie break preserves the exact maximum.\n    winner = min(best, key=lambda i: (abs(thresholds[i] - 0.5), -thresholds[i]))\n    return float(thresholds[winner]), float(scores[winner])\n\n\ndef fit_calibrator(y_true, neural_probs, sparse_probs):\n    """Select each label\'s blend and threshold using supplied training labels."""\n    truth, neural, sparse = _inputs(y_true, neural_probs, sparse_probs)\n    weights, thresholds = [], []\n    for column in range(len(TARGET_SYMPTOMS)):\n        candidates = []\n        for weight in WEIGHT_GRID:\n            blended = weight * neural[:, column] + (1.0 - weight) * sparse[:, column]\n            threshold, score = _best_threshold(truth[:, column], blended)\n            candidates.append((score, -abs(weight - 0.5), weight, threshold))\n        _, _, weight, threshold = max(candidates)\n        weights.append(float(weight))\n        thresholds.append(float(threshold))\n    return {"neural_weights": weights, "thresholds": thresholds}\n\n\ndef apply_calibrator(neural_probs, sparse_probs, calibrator):\n    """Return blended probabilities; use >= stored thresholds for labels."""\n    neural = _probabilities(neural_probs, "neural_probs")\n    sparse = _probabilities(sparse_probs, "sparse_probs")\n    if neural.shape != sparse.shape:\n        raise ValueError("neural_probs and sparse_probs must have identical shapes")\n    weights = np.asarray(calibrator["neural_weights"], dtype=np.float64)\n    thresholds = np.asarray(calibrator["thresholds"], dtype=np.float64)\n    expected = (len(TARGET_SYMPTOMS),)\n    if weights.shape != expected or thresholds.shape != expected:\n        raise ValueError("calibrator must contain one weight and threshold per symptom")\n    if not np.isfinite(weights).all() or np.any((weights < 0) | (weights > 1)):\n        raise ValueError("calibrator weights must be finite and in [0, 1]")\n    if not np.isfinite(thresholds).all() or np.any((thresholds < 0) | (thresholds > ALL_NEGATIVE_THRESHOLD)):\n        raise ValueError("calibrator thresholds must be finite and in [0, nextafter(1, inf)]")\n    return weights * neural + (1.0 - weights) * sparse\n\n\ndef _grouped_splits(truth, groups, seed, n_splits):\n    groups = np.asarray(groups)\n    if groups.ndim != 1 or len(groups) != len(truth):\n        raise ValueError("groups must contain one group ID per sample")\n    if not isinstance(n_splits, (int, np.integer)) or n_splits < 2:\n        raise ValueError("n_splits must be an integer >= 2")\n    unique_groups, inverse = np.unique(groups, return_inverse=True)\n    if len(unique_groups) < 2:\n        raise ValueError("cross-fit calibration requires at least two distinct groups")\n    n_splits = min(n_splits, len(unique_groups))\n    group_targets = np.zeros((len(unique_groups), truth.shape[1]), dtype=np.int8)\n    np.maximum.at(group_targets, inverse, truth)\n    splitter = MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)\n    for train_groups, test_groups in splitter.split(np.zeros(len(unique_groups)), group_targets):\n        train = np.flatnonzero(np.isin(inverse, train_groups))\n        test = np.flatnonzero(np.isin(inverse, test_groups))\n        if not len(train) or not len(test):\n            raise ValueError("grouped calibration produced an empty fold")\n        yield train, test\n\n\ndef calibrate_crossfit(y_true, neural_probs, sparse_probs, groups, seed=42, n_splits=5):\n    """Cross-fit within Training calibration data, then fit deployment settings.\n\n    Base-model predictions must themselves come from samples excluded from\n    base-model training and model selection. Never pass official Validation.\n    """\n    truth, neural, sparse = _inputs(y_true, neural_probs, sparse_probs)\n    crossfit_predictions = np.zeros_like(truth)\n    seen = np.zeros(len(truth), dtype=np.int8)\n    fold_counts = []\n    for train, test in _grouped_splits(truth, groups, seed, n_splits):\n        calibrator = fit_calibrator(truth[train], neural[train], sparse[train])\n        probabilities = apply_calibrator(neural[test], sparse[test], calibrator)\n        crossfit_predictions[test] = probabilities >= np.asarray(calibrator["thresholds"])\n        seen[test] += 1\n        fold_counts.append({"fit_samples": int(len(train)), "score_samples": int(len(test))})\n    if not np.all(seen == 1):\n        raise RuntimeError("each calibration sample must be scored exactly once")\n    final_calibrator = fit_calibrator(truth, neural, sparse)\n    fitted = apply_calibrator(neural, sparse, final_calibrator) >= np.asarray(final_calibrator["thresholds"])\n    precision, recall, f1, support = precision_recall_fscore_support(\n        truth, crossfit_predictions, average=None, zero_division=0\n    )\n    report = {\n        "score_scope": "Training-only calibration; cross-fit estimates calibration generalization, not official Validation.",\n        "calibration_fit_macro_f1": float(f1_score(truth, fitted, average="macro", zero_division=0)),\n        "calibration_crossfit_macro_f1": float(f1.mean()),\n        "per_symptom_crossfit": {\n            symptom: {"precision": float(p), "recall": float(r), "f1": float(f), "support": int(s)}\n            for symptom, p, r, f, s in zip(TARGET_SYMPTOMS, precision, recall, f1, support)\n        },\n        "calibration_samples": int(len(truth)),\n        "calibration_groups": int(len(np.unique(groups))),\n        "crossfit_splits": len(fold_counts),\n        "crossfit_fold_counts": fold_counts,\n    }\n    return final_calibrator, report\n', 'neural.py': '"""Document-level multi-label transformer; every dialogue window is retained."""\n\nfrom __future__ import annotations\n\nimport math\nimport os\nimport random\nimport re\nimport tempfile\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom sklearn.metrics import average_precision_score\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset\nfrom transformers import AutoConfig, AutoModel, get_linear_schedule_with_warmup\n\n\nNUM_LABELS = 9\n\n\ndef tokenize_dialogues(texts, tokenizer, max_length=512, stride=64):\n    """Return documents of token windows, including the last window of long calls.\n\n    Only ordinary tokenizer special tokens are used. The stride is an overlap in\n    tokens, not an upper bound on the number of windows or dialogue length.\n    """\n    usable_length = max_length - tokenizer.num_special_tokens_to_add(pair=False)\n    if usable_length < 1 or not 0 <= stride < usable_length:\n        raise ValueError("max_length must fit special tokens; 0 <= stride < usable length")\n    texts = list(texts)\n    documents = [[] for _ in texts]\n    for start in range(0, len(texts), 128):\n        encoded = tokenizer(\n            texts[start:start + 128], truncation=True, max_length=max_length,\n            stride=stride, return_overflowing_tokens=True, padding=False,\n            return_attention_mask=True,\n        )\n        if "overflow_to_sample_mapping" not in encoded:\n            raise ValueError("Use a fast tokenizer (AutoTokenizer(..., use_fast=True))")\n        for window, sample in enumerate(encoded["overflow_to_sample_mapping"]):\n            documents[start + int(sample)].append({\n                key: encoded[key][window]\n                for key in tokenizer.model_input_names if key in encoded\n            })\n    if any(not windows for windows in documents):\n        raise ValueError("Tokenizer returned a document without any windows")\n    return documents\n\n\nclass DialogueDataset(Dataset):\n    def __init__(self, documents, targets=None):\n        self.documents, self.targets = documents, targets\n\n    def __len__(self):\n        return len(self.documents)\n\n    def __getitem__(self, index):\n        item = {"windows": self.documents[index]}\n        if self.targets is not None:\n            item["labels"] = self.targets[index]\n        return item\n\n\nclass DialogueCollator:\n    """Flatten windows for the encoder while keeping exactly one target per call."""\n\n    def __init__(self, tokenizer):\n        self.tokenizer = tokenizer\n\n    def __call__(self, examples):\n        windows, doc_indices = [], []\n        for index, example in enumerate(examples):\n            if not example["windows"]:\n                raise ValueError("A document must contain at least one token window")\n            windows.extend(example["windows"])\n            doc_indices.extend([index] * len(example["windows"]))\n        batch = dict(self.tokenizer.pad(windows, padding=True, return_tensors="pt"))\n        batch["doc_indices"] = torch.tensor(doc_indices, dtype=torch.long)\n        if "labels" in examples[0]:\n            batch["labels"] = torch.from_numpy(np.asarray(\n                [example["labels"] for example in examples], dtype=np.float32,\n            ))\n        return batch\n\n\nclass DialogueClassifier(nn.Module):\n    """Label-specific attention pools complete dialogues before computing BCE."""\n\n    def __init__(self, encoder, num_labels=NUM_LABELS, dropout=0.2,\n                 window_batch_size=8):\n        super().__init__()\n        self.encoder = encoder\n        self.window_batch_size = int(window_batch_size)\n        if self.window_batch_size < 1:\n            raise ValueError("window_batch_size must be positive")\n        hidden = encoder.config.hidden_size\n        self.projection = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.Tanh())\n        self.dropout = nn.Dropout(dropout)\n        self.attention_queries = nn.Parameter(torch.empty(num_labels, hidden))\n        self.label_weights = nn.Parameter(torch.empty(num_labels, hidden))\n        self.label_bias = nn.Parameter(torch.zeros(num_labels))\n        nn.init.normal_(self.attention_queries, std=0.02)\n        nn.init.normal_(self.label_weights, std=0.02)\n\n    def forward(self, doc_indices, **inputs):\n        representations = []\n        for start in range(0, len(doc_indices), self.window_batch_size):\n            window_inputs = {key: value[start:start + self.window_batch_size]\n                             for key, value in inputs.items()}\n            hidden = self.encoder(**window_inputs).last_hidden_state\n            mask = window_inputs["attention_mask"].unsqueeze(-1).to(hidden.dtype)\n            mean = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)\n            representations.append(self.projection(torch.cat([hidden[:, 0], mean], -1)))\n        windows = self.dropout(torch.cat(representations))\n        logits = []\n        # Each label learns which part of a potentially long call provides evidence.\n        for document_index in range(int(doc_indices.max().item()) + 1):\n            selected = windows[doc_indices == document_index]\n            scores = selected @ self.attention_queries.T / math.sqrt(selected.shape[-1])\n            attention = torch.softmax(scores.float(), dim=0).to(selected.dtype)\n            pooled = attention.T @ selected\n            logits.append((self.dropout(pooled) * self.label_weights).sum(-1) + self.label_bias)\n        return torch.stack(logits)\n\n\ndef _device(config):\n    device = torch.device(config.get("device", "cuda" if torch.cuda.is_available() else "cpu"))\n    if device.type == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("CUDA was requested but is unavailable. Select a GPU runtime.")\n    return device\n\n\ndef _amp(device):\n    dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16\n    return torch.amp.autocast(device_type=device.type, dtype=dtype, enabled=device.type == "cuda")\n\n\ndef _loader(documents, targets, tokenizer, batch_size, shuffle=False, seed=0):\n    if int(batch_size) < 1:\n        raise ValueError("batch_size must be positive")\n    generator = torch.Generator().manual_seed(seed)\n    return DataLoader(\n        DialogueDataset(documents, targets), batch_size=int(batch_size),\n        shuffle=shuffle, collate_fn=DialogueCollator(tokenizer),\n        num_workers=0, generator=generator,\n    )\n\n\n@torch.inference_mode()\ndef _predict(model, loader, device):\n    model.eval()\n    probabilities = []\n    for batch in loader:\n        batch.pop("labels", None)\n        batch = {key: value.to(device) for key, value in batch.items()}\n        with _amp(device):\n            logits = model(**batch)\n        probabilities.append(logits.float().sigmoid().cpu().numpy())\n    return np.concatenate(probabilities) if probabilities else np.empty((0, NUM_LABELS), dtype=np.float32)\n\n\ndef _optimizer(model, config):\n    depth = int(getattr(model.encoder.config, "num_hidden_layers", 0))\n    decay = float(config.get("layerwise_lr_decay", 0.9))\n    if not 0 < decay <= 1:\n        raise ValueError("layerwise_lr_decay must be in (0, 1]")\n    groups = {}\n    for name, parameter in model.named_parameters():\n        if not parameter.requires_grad:\n            continue\n        if name.startswith("encoder."):\n            match = re.search(r"\\.layer\\.(\\d+)\\.", name)\n            level = int(match.group(1)) + 1 if match else (0 if ".embeddings." in name else depth + 1)\n            lr = float(config.get("learning_rate", 1.5e-5)) * decay ** (depth + 1 - level)\n        else:\n            lr = float(config.get("head_learning_rate", 1e-4))\n        wd = 0.0 if parameter.ndim < 2 or name.endswith("bias") else float(config.get("weight_decay", 0.01))\n        groups.setdefault((lr, wd), []).append(parameter)\n    return torch.optim.AdamW([{"params": params, "lr": lr, "weight_decay": wd}\n                             for (lr, wd), params in groups.items()])\n\n\ndef _atomic_save(checkpoint, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    descriptor, temporary = tempfile.mkstemp(prefix=path.name + ".", suffix=".tmp", dir=path.parent)\n    os.close(descriptor)\n    try:\n        torch.save(checkpoint, temporary)\n        os.replace(temporary, path)\n    finally:\n        if os.path.exists(temporary):\n            os.unlink(temporary)\n\n\ndef _targets(targets, documents, name):\n    targets = np.asarray(targets, dtype=np.float32)\n    if targets.shape != (len(documents), NUM_LABELS) or not len(documents):\n        raise ValueError(f"{name} requires nonempty documents and targets of shape (N, {NUM_LABELS})")\n    if not np.isin(targets, [0, 1]).all():\n        raise ValueError(f"{name} targets must contain only 0 and 1")\n    return targets\n\n\ndef train_model(train_documents, train_targets, stop_documents, stop_targets,\n                tokenizer, config: dict, checkpoint_path: Path, seed: int):\n    """Train on documents; select best epoch only by stopping-set macro AP.\n\n    This function does not tune thresholds and never uses calibration targets.\n    GPU enforcement for production belongs to the orchestration entry point.\n    """\n    train_targets = _targets(train_targets, train_documents, "train")\n    stop_targets = _targets(stop_targets, stop_documents, "stop")\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    device = _device(config)\n    max_epochs, patience = int(config.get("max_epochs", 8)), int(config.get("patience", 2))\n    accumulation = int(config.get("gradient_accumulation_steps", 4))\n    if min(max_epochs, patience, accumulation) < 1:\n        raise ValueError("max_epochs, patience and gradient_accumulation_steps must be positive")\n    model_name = config.get("model_name", "klue/roberta-large")\n    encoder = AutoModel.from_pretrained(model_name)\n    if config.get("gradient_checkpointing", True):\n        encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n    if hasattr(encoder.config, "use_cache"):\n        encoder.config.use_cache = False\n    architecture = {"num_labels": NUM_LABELS, "dropout": float(config.get("dropout", 0.2)),\n                    "window_batch_size": int(config.get("window_batch_size", 8))}\n    model = DialogueClassifier(encoder, **architecture).to(device)\n    train_loader = _loader(train_documents, train_targets, tokenizer,\n                           config.get("batch_size", 2), shuffle=True, seed=seed)\n    stop_loader = _loader(stop_documents, None, tokenizer, config.get("inference_batch_size", 4))\n    positives = train_targets.sum(axis=0)\n    if (positives == 0).any():\n        raise ValueError(f"Training split has no positive examples for labels {np.flatnonzero(positives == 0).tolist()}")\n    pos_weight = np.clip(((len(train_targets) - positives) / positives).clip(min=1)\n                         ** float(config.get("pos_weight_power", 0.5)), 1, 5)\n    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.as_tensor(pos_weight, device=device))\n    optimizer = _optimizer(model, config)\n    total_steps = math.ceil(len(train_loader) / accumulation) * max_epochs\n    scheduler = get_linear_schedule_with_warmup(\n        optimizer, num_warmup_steps=int(total_steps * float(config.get("warmup_ratio", 0.1))),\n        num_training_steps=total_steps,\n    )\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and not torch.cuda.is_bf16_supported())\n    valid_labels = stop_targets.sum(axis=0) > 0\n    if not valid_labels.any():\n        raise ValueError("Stopping split has no positive labels; macro AP cannot select an epoch")\n    if not valid_labels.all():\n        print(f"Warning: stop AP excludes labels without positives: {np.flatnonzero(~valid_labels).tolist()}", flush=True)\n    best_ap, best_epoch, stale, history = -1.0, 0, 0, []\n    for epoch in range(1, max_epochs + 1):\n        epoch_started = time.monotonic()\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n        loss_sum, document_count = 0.0, 0\n        for step, batch in enumerate(train_loader):\n            labels = batch.pop("labels").to(device)\n            batch = {key: value.to(device) for key, value in batch.items()}\n            # Match a single full document batch, including a shorter last batch.\n            group_start = (step // accumulation) * accumulation\n            group_documents = min(accumulation * train_loader.batch_size,\n                                  len(train_documents) - group_start * train_loader.batch_size)\n            with _amp(device):\n                loss = criterion(model(**batch).float(), labels)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(f"Non-finite training loss at epoch={epoch}, batch={step}")\n            scaler.scale(loss * len(labels) / group_documents).backward()\n            loss_sum += loss.detach().item() * len(labels)\n            document_count += len(labels)\n            if (step + 1) % accumulation == 0 or step + 1 == len(train_loader):\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                old_scale = scaler.get_scale()\n                scaler.step(optimizer)\n                scaler.update()\n                if scaler.get_scale() >= old_scale:\n                    scheduler.step()\n                optimizer.zero_grad(set_to_none=True)\n            log_every = int(config.get("log_every", 100))\n            if log_every > 0 and (step + 1) % log_every == 0:\n                elapsed = (time.monotonic() - epoch_started) / 60\n                print(f"seed={seed} epoch={epoch} batch={step + 1}/{len(train_loader)} "\n                      f"loss={loss_sum / document_count:.5f} minutes={elapsed:.1f}", flush=True)\n        probabilities = _predict(model, stop_loader, device)\n        if not np.isfinite(probabilities).all():\n            raise FloatingPointError("Stopping predictions contain non-finite values")\n        score = float(average_precision_score(stop_targets[:, valid_labels],\n                                              probabilities[:, valid_labels], average="macro"))\n        row = {"epoch": epoch, "train_loss": loss_sum / document_count,\n               "stop_macro_ap": score, "stop_labels_with_positives": int(valid_labels.sum()),\n               "minutes": (time.monotonic() - epoch_started) / 60}\n        history.append(row)\n        print(f"seed={seed} epoch={epoch}/{max_epochs} loss={row[\'train_loss\']:.5f} "\n              f"stop_AP={score:.5f} minutes={row[\'minutes\']:.1f}", flush=True)\n        if score > best_ap + 1e-6:\n            best_ap, best_epoch, stale = score, epoch, 0\n            _atomic_save({\n                "format_version": 1, "encoder_config": encoder.config.to_dict(),\n                "architecture": architecture, "model_name": str(model_name),\n                "model_state": {key: value.detach().cpu() for key, value in model.state_dict().items()},\n                "epoch": epoch, "stop_ap": score, "seed": int(seed),\n            }, checkpoint_path)\n        else:\n            stale += 1\n            if stale >= patience:\n                break\n    return {"best_epoch": best_epoch, "best_stop_ap": best_ap, "history": history}\n\n\ndef predict_checkpoint(checkpoint_path, documents, tokenizer, config):\n    """Rebuild from the saved config and weights; no pretrained network fetch."""\n    device = _device(config)\n    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    if checkpoint.get("format_version") != 1:\n        raise ValueError(f"Unsupported neural checkpoint format: {checkpoint_path}")\n    encoder_data = dict(checkpoint["encoder_config"])\n    model_type = encoder_data.pop("model_type")\n    encoder_config = AutoConfig.for_model(model_type, **encoder_data)\n    architecture = dict(checkpoint["architecture"])\n    architecture["window_batch_size"] = int(config.get("window_batch_size", architecture["window_batch_size"]))\n    model = DialogueClassifier(AutoModel.from_config(encoder_config), **architecture)\n    model.load_state_dict(checkpoint["model_state"], strict=True)\n    del checkpoint\n    model.to(device)\n    loader = _loader(documents, None, tokenizer, config.get("inference_batch_size", 4))\n    predictions = _predict(model, loader, device)\n    if not np.isfinite(predictions).all():\n        raise FloatingPointError("Inference predictions contain non-finite values")\n    return predictions\n', 'pipeline.py': '"""Training-only model selection, ensemble calibration, and portable inference."""\nfrom __future__ import annotations\n\nimport ast\nimport csv\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport shutil\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport torch\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import FeatureUnion\nfrom transformers import AutoTokenizer\n\nif __package__:\n    from .calibration import apply_calibrator, calibrate_crossfit\n    from .data import TARGET_SYMPTOMS, load_dataset, make_splits, score_predictions, write_json, write_predictions\n    from .neural import predict_checkpoint, tokenize_dialogues, train_model\nelse:\n    from calibration import apply_calibrator, calibrate_crossfit\n    from data import TARGET_SYMPTOMS, load_dataset, make_splits, score_predictions, write_json, write_predictions\n    from neural import predict_checkpoint, tokenize_dialogues, train_model\n\n\nDEFAULT_CONFIG = {\n    "model_name": "klue/roberta-large",\n    "seed": 42,\n    "n_folds": 5,\n    "calibration_size": 0.10,\n    "calibration_folds": 5,\n    "max_length": 512,\n    "stride": 64,\n    "max_epochs": 6,\n    "patience": 2,\n    "batch_size": 2,\n    "inference_batch_size": 8,\n    "window_batch_size": 2,\n    "gradient_accumulation_steps": 16,\n    "learning_rate": 1.5e-5,\n    "head_learning_rate": 1e-4,\n    "layerwise_lr_decay": 0.95,\n    "weight_decay": 0.01,\n    "warmup_ratio": 0.10,\n    "dropout": 0.20,\n    "pos_weight_power": 0.5,\n    "gradient_checkpointing": True,\n    "device": "cuda",\n    "require_cuda": True,\n    "char_features": 120_000,\n    "word_features": 40_000,\n    "min_df": 2,\n    "logistic_c": 4.0,\n}\nRUNTIME_FILES = ("data.py", "calibration.py", "neural.py", "pipeline.py", "train.py", "inference.py", "evaluate.py")\n\n\ndef fit_sparse(texts, targets, config):\n    """문자 표현과 단어 순서를 보완하는, Training에만 fit한 TF-IDF 모델."""\n    features = FeatureUnion([\n        ("char", TfidfVectorizer(analyzer="char", ngram_range=(2, 5),\n                                max_features=config["char_features"], min_df=config["min_df"],\n                                sublinear_tf=True, dtype=np.float32)),\n        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), token_pattern=r"(?u)\\b\\w+\\b",\n                                max_features=config["word_features"], min_df=config["min_df"],\n                                sublinear_tf=True, dtype=np.float32)),\n    ])\n    matrix = features.fit_transform(texts)\n    classifiers = []\n    for column, name in enumerate(TARGET_SYMPTOMS):\n        truth = targets[:, column]\n        if len(np.unique(truth)) == 1:\n            classifier = float(truth[0])\n        else:\n            classifier = LogisticRegression(C=config["logistic_c"], class_weight="balanced",\n                                            solver="liblinear", max_iter=1500, random_state=config["seed"])\n            classifier.fit(matrix, truth)\n        classifiers.append(classifier)\n        print(f"TF-IDF fit: {name}", flush=True)\n    return {"features": features, "classifiers": classifiers}\n\n\ndef predict_sparse(model, texts):\n    if not texts:\n        return np.empty((0, len(TARGET_SYMPTOMS)), dtype=np.float32)\n    # 큰 Test 폴더도 전체 sparse matrix를 RAM에 동시에 올리지 않는다.\n    batches = []\n    for start in range(0, len(texts), 1024):\n        matrix = model["features"].transform(texts[start:start + 1024])\n        columns = [np.full(matrix.shape[0], clf) if isinstance(clf, float)\n                   else clf.predict_proba(matrix)[:, 1] for clf in model["classifiers"]]\n        batches.append(np.column_stack(columns))\n    return np.concatenate(batches).astype(np.float32)\n\n\ndef _release_memory():\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\ndef _torch_save(path, value):\n    path = Path(path)\n    temporary = path.with_name(path.name + ".tmp")\n    torch.save(value, temporary)\n    temporary.replace(path)\n\n\ndef _fingerprint(paths, texts, targets, config):\n    digest = hashlib.sha256(json.dumps(config, sort_keys=True).encode())\n    for path, text, target in zip(paths, texts, targets):\n        digest.update(json.dumps([path.name, text, target.tolist()], ensure_ascii=False).encode("utf-8"))\n    # 변경된 코드로 기존 fold가 조용히 재사용되는 것을 방지한다.\n    for name in RUNTIME_FILES:\n        path = Path(__file__).parent / name\n        digest.update(path.name.encode())\n        digest.update(path.read_bytes())\n    return digest.hexdigest()\n\n\ndef train_pipeline(train_label_dir, output_dir, config=None):\n    """Validation 인자를 받지 않는다. 보정 후에도 모델을 재학습하지 않는다."""\n    supplied = config or {}\n    unknown = set(supplied) - set(DEFAULT_CONFIG)\n    if unknown:\n        raise ValueError(f"Unknown config keys: {sorted(unknown)}")\n    config = {**DEFAULT_CONFIG, **supplied}\n    train_label_dir, output_dir = Path(train_label_dir).resolve(), Path(output_dir).resolve()\n    if train_label_dir.name != "TL_서울_구급" or "Validation" in train_label_dir.parts:\n        raise ValueError("학습 입력은 서울 Training의 TL_서울_구급 폴더여야 합니다.")\n    if config["require_cuda"] and (config["device"] != "cuda" or not torch.cuda.is_available()):\n        raise RuntimeError("전체 학습에는 GPU가 필요합니다. Colab GPU 런타임에서 실행하세요.")\n    if config["device"] == "cuda":\n        torch.set_float32_matmul_precision("high")\n        print("GPU:", torch.cuda.get_device_name(0), flush=True)\n    paths, texts, targets = load_dataset(train_label_dir, include_targets=True)\n    dev, calibration, folds, groups = make_splits(\n        texts, targets, config["calibration_size"], config["n_folds"], config["seed"],\n    )\n    output_dir.mkdir(parents=True, exist_ok=True)\n    signature = _fingerprint(paths, texts, targets, config)\n    run_path = output_dir / "run.json"\n    if run_path.exists() and json.loads(run_path.read_text(encoding="utf-8"))["signature"] != signature:\n        raise ValueError("출력 폴더에 다른 데이터/설정/코드의 실험이 있습니다. 새 output_dir를 지정하세요.")\n    write_json(run_path, {"signature": signature, "config": config, "training_samples": len(paths)})\n    write_json(output_dir / "splits.json", {\n        "calibration": [paths[i].name for i in calibration],\n        "folds": [{"fit": [paths[i].name for i in fit], "stop": [paths[i].name for i in stop]}\n                  for fit, stop in folds],\n        "seed": config["seed"], "grouping": "SHA256 of whitespace-normalized transcript only",\n    })\n    print(f"Training={len(paths):,}, development={len(dev):,}, calibration={len(calibration):,}", flush=True)\n    tokenizer_dir = output_dir / "tokenizer"\n    if (tokenizer_dir / "tokenizer_config.json").exists():\n        tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, use_fast=True, local_files_only=True)\n    else:\n        tokenizer = AutoTokenizer.from_pretrained(config["model_name"], use_fast=True)\n        tokenizer.save_pretrained(tokenizer_dir)\n    documents = tokenize_dialogues(texts, tokenizer, config["max_length"], config["stride"])\n    calibration_documents = [documents[i] for i in calibration]\n    counts = np.asarray([len(document) for document in documents])\n    print(f"대화 window 수: 평균={counts.mean():.2f}, 최대={counts.max()}, 긴 대화={int((counts > 1).sum()):,}", flush=True)\n    neural_probs = np.zeros((len(calibration), len(TARGET_SYMPTOMS)), dtype=np.float64)\n    fold_names, histories = [], []\n    for number, (fit, stop) in enumerate(folds, 1):\n        name = f"fold_{number}.ckpt"\n        checkpoint_path, done_path = output_dir / name, output_dir / f"fold_{number}_done.json"\n        print(f"\\nFold {number}/{len(folds)}: fit={len(fit):,}, stop={len(stop):,}", flush=True)\n        if done_path.exists() and checkpoint_path.exists():\n            result = json.loads(done_path.read_text(encoding="utf-8"))\n            print("완료된 fold 재사용 (동일 run signature 확인됨)", flush=True)\n        else:\n            result = train_model([documents[i] for i in fit], targets[fit],\n                                 [documents[i] for i in stop], targets[stop], tokenizer,\n                                 config, checkpoint_path, seed=config["seed"] + number * 1009)\n            write_json(done_path, result)\n        _release_memory()\n        neural_probs += predict_checkpoint(checkpoint_path, calibration_documents, tokenizer, config)\n        _release_memory()\n        fold_names.append(name)\n        histories.append(result)\n    neural_probs /= len(folds)\n    sparse_path = output_dir / "sparse.joblib"\n    if sparse_path.exists():\n        sparse = joblib.load(sparse_path)\n    else:\n        sparse = fit_sparse([texts[i] for i in dev], targets[dev], config)\n        temporary = sparse_path.with_name("sparse.joblib.tmp")\n        joblib.dump(sparse, temporary, compress=3)\n        temporary.replace(sparse_path)\n    sparse_probs = predict_sparse(sparse, [texts[i] for i in calibration])\n    del sparse\n    calibrator, report = calibrate_crossfit(targets[calibration], neural_probs, sparse_probs,\n                                          groups[calibration], seed=config["seed"] + 2,\n                                          n_splits=config["calibration_folds"])\n    # OOF 확률을 final refit에 전용하지 않고, 실제 배포 ensemble의 확률을 보정한다.\n    np.savez_compressed(output_dir / "calibration_predictions.npz", targets=targets[calibration],\n                        neural=neural_probs, sparse=sparse_probs,\n                        filenames=np.asarray([paths[i].name for i in calibration]))\n    versions = {name: importlib.metadata.version(name) for name in\n                ("torch", "transformers", "numpy", "scikit-learn", "iterative-stratification")}\n    manifest = {"format": "mission3_ver4", "version": 1, "target_symptoms": list(TARGET_SYMPTOMS),\n                "config": config, "folds": fold_names, "tokenizer": "tokenizer", "sparse": "sparse.joblib",\n                "calibrator": calibrator, "signature": signature, "versions": versions}\n    checkpoint_path = output_dir / "mission3_ver4_model.ckpt"\n    _torch_save(checkpoint_path, manifest)\n    report.update({"training_samples": len(paths), "development_samples": len(dev),\n                   "model": f"{config[\'model_name\']} {len(folds)}-fold ensemble + char/word TF-IDF LR",\n                   "calibrator": calibrator, "fold_best_epochs": [x["best_epoch"] for x in histories],\n                   "versions": versions, "validation_used_for_training_or_calibration": False})\n    write_json(output_dir / "mission3_ver4_metrics.json", report)\n    write_json(output_dir / "mission3_ver4_history.json", histories)\n    # 노트북 하나만 Colab에 올려도 제출 코드와 실행 당시 설정을 남긴다.\n    submission = output_dir / "submission"\n    submission.mkdir(exist_ok=True)\n    for name in (*RUNTIME_FILES, "requirements.txt", "README.md"):\n        path = Path(__file__).parent / name\n        if path.exists():\n            shutil.copy2(path, submission / path.name)\n    print(json.dumps(report, ensure_ascii=False, indent=2), flush=True)\n    print("Checkpoint:", checkpoint_path, flush=True)\n    return checkpoint_path, report\n\n\ndef load_manifest(checkpoint_path):\n    manifest = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    if manifest.get("format") != "mission3_ver4" or manifest.get("version") != 1:\n        raise ValueError("지원하지 않는 checkpoint입니다. ver4의 manifest .ckpt를 지정하세요.")\n    if manifest["target_symptoms"] != list(TARGET_SYMPTOMS) or not manifest["folds"]:\n        raise ValueError("Checkpoint class order/folds mismatch")\n    root = Path(checkpoint_path).resolve().parent\n    for name in [*manifest["folds"], manifest["sparse"], manifest["tokenizer"]]:\n        path = (root / name).resolve()\n        if not path.is_relative_to(root) or not path.exists():\n            raise FileNotFoundError(f"Checkpoint 구성 파일을 함께 보관해야 합니다: {name}")\n    return manifest\n\n\ndef predict_pipeline(checkpoint_path, label_dir, output, device=None, batch_size=None):\n    """저장된 encoder config/tokenizer만 사용하므로 학습 후 오프라인 추론 가능."""\n    checkpoint_path = Path(checkpoint_path).resolve()\n    manifest = load_manifest(checkpoint_path)\n    root = checkpoint_path.parent\n    config = dict(manifest["config"])\n    config["device"] = device or ("cuda" if torch.cuda.is_available() else "cpu")\n    if batch_size is not None:\n        config["inference_batch_size"] = batch_size\n    paths, texts, _ = load_dataset(Path(label_dir), include_targets=False)\n    tokenizer = AutoTokenizer.from_pretrained(root / manifest["tokenizer"], use_fast=True, local_files_only=True)\n    documents = tokenize_dialogues(texts, tokenizer, config["max_length"], config["stride"])\n    neural_probs = np.zeros((len(paths), len(TARGET_SYMPTOMS)), dtype=np.float64)\n    for name in manifest["folds"]:\n        print("Predict:", name, flush=True)\n        neural_probs += predict_checkpoint(root / name, documents, tokenizer, config)\n        _release_memory()\n    neural_probs /= len(manifest["folds"])\n    sparse = joblib.load(root / manifest["sparse"])\n    sparse_probs = predict_sparse(sparse, texts)\n    probabilities = apply_calibrator(neural_probs, sparse_probs, manifest["calibrator"])\n    predictions = write_predictions(paths, probabilities, manifest["calibrator"]["thresholds"], Path(output))\n    print(f"{len(paths):,} predictions saved: {output}", flush=True)\n    return paths, probabilities, predictions\n\n\ndef evaluate_csv(label_dir, prediction_path):\n    """이미 저장된 예측을 파일명으로 정답에 연결한다. 모델/임계값 변경 없음."""\n    with Path(prediction_path).open(encoding="utf-8-sig", newline="") as handle:\n        reader = csv.DictReader(handle)\n        if reader.fieldnames != ["label file name", "symptom"]:\n            raise ValueError("CSV header must be: label file name,symptom")\n        rows = list(reader)\n    by_name = {}\n    for row in rows:\n        if row["label file name"] in by_name:\n            raise ValueError("Duplicate prediction filename")\n        labels = ast.literal_eval(row["symptom"])\n        if not isinstance(labels, list) or any(x not in TARGET_SYMPTOMS for x in labels) or len(set(labels)) != len(labels):\n            raise ValueError("CSV symptom must be a list of unique target strings")\n        by_name[row["label file name"]] = [int(x in labels) for x in TARGET_SYMPTOMS]\n    paths, _, targets = load_dataset(Path(label_dir), include_targets=True)\n    if set(by_name) != {path.name for path in paths}:\n        raise ValueError("CSV와 JSON 파일명 집합이 일치하지 않습니다.")\n    return score_predictions(targets, np.asarray([by_name[path.name] for path in paths], dtype=np.int8))\n', 'runner.py': '"""Frozen ver4 ensemble inference; no model fitting or calibration search."""\nimport csv\nimport importlib.metadata\nimport json\nimport os\nimport sys\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\n# All encoders and the tokenizer must come from the saved bundle.\nos.environ["HF_HUB_OFFLINE"] = "1"\nos.environ["TRANSFORMERS_OFFLINE"] = "1"\nos.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\n\nimport torch\nimport pipeline as runtime_pipeline\nimport data as runtime_data\n\n\ndef main(config_path):\n    config = json.loads(Path(config_path).read_text(encoding="utf-8"))\n    checkpoint_path = Path(config["checkpoint_path"])\n    label_dir = Path(config["label_dir"])\n    output_dir = Path(config["output_dir"])\n    prediction_path = output_dir / "mission3.csv"\n    # Prevent overwriting a completed run even when just this cell is rerun.\n    if any((output_dir / name).exists() for name in\n           ("mission3.csv", "validation_metrics.json", "inference_summary.json")):\n        raise FileExistsError(f"이미 결과가 있습니다. 1번 셀부터 새 출력 폴더로 실행하세요: {output_dir}")\n    manifest = runtime_pipeline.load_manifest(checkpoint_path)\n    device = config["device"]\n    if device == "auto":\n        device = "cuda" if torch.cuda.is_available() else "cpu"\n    if device == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("CUDA GPU를 사용할 수 없습니다. GPU 런타임을 연결하거나 DEVICE=\'cpu\'로 실행하세요.")\n    versions = {name: importlib.metadata.version(name) for name in\n                ("torch", "transformers", "numpy", "scikit-learn", "scipy", "joblib", "iterative-stratification")}\n    for name in ("transformers", "numpy", "scikit-learn"):\n        saved = manifest.get("versions", {}).get(name)\n        if saved and versions[name] != saved:\n            raise RuntimeError(f"저장 모델과 {name} 버전이 다릅니다: 저장={saved}, 현재={versions[name]}. 환경 준비 셀을 실행하세요.")\n\n    best_epochs = []\n    for fold in manifest["folds"]:\n        history_path = checkpoint_path.parent / (Path(fold).stem + "_done.json")\n        epoch = json.loads(history_path.read_text(encoding="utf-8")).get("best_epoch") if history_path.is_file() else None\n        best_epochs.append(epoch)\n    print("실행 장치:", torch.cuda.get_device_name(0) if device == "cuda" else "CPU (GPU보다 오래 걸릴 수 있습니다)", flush=True)\n    print("저장 모델:", len(manifest["folds"]), "개 | 기록된 최적 epoch:", best_epochs, flush=True)\n    print("epoch는 저장된 가중치에 이미 반영되어 있으며 재학습하지 않습니다.", flush=True)\n    print("증상 | 신경망 가중치 | TF-IDF 가중치 | 임계값", flush=True)\n    calibrator = manifest["calibrator"]\n    for name, weight, threshold in zip(manifest["target_symptoms"], calibrator["neural_weights"], calibrator["thresholds"]):\n        print(f"{name} | {weight:.2f} | {1 - weight:.2f} | {threshold:.9f}", flush=True)\n    print("위 표시는 반올림이며, 계산에는 manifest에 저장된 원래 정밀도의 값을 적용합니다.", flush=True)\n\n    started_at = datetime.now(timezone.utc).isoformat()\n    started = time.monotonic()\n    paths, probabilities, predictions = runtime_pipeline.predict_pipeline(\n        checkpoint_path, label_dir, prediction_path, device=device, batch_size=config["batch_size"],\n    )\n    inference_seconds = time.monotonic() - started\n\n    # Read labels only after the complete prediction CSV has been saved.\n    metrics = None\n    labeled_count = None\n    evaluation_status = "disabled"\n    if config["run_evaluation"]:\n        labeled_count = 0\n        for path in paths:\n            annotation = json.loads(path.read_text(encoding="utf-8-sig"))\n            labeled_count += int("symptom" in annotation)\n        if labeled_count == len(paths):\n            metrics = runtime_pipeline.evaluate_csv(label_dir, prediction_path)\n            runtime_data.write_json(output_dir / "validation_metrics.json", metrics)\n            evaluation_status = "completed"\n            print(f"Macro F1: {metrics[\'macro_f1\']:.6f} ({metrics[\'samples\']:,}개)", flush=True)\n            for name, values in metrics["per_symptom"].items():\n                print(f"  {name}: F1={values[\'f1\']:.6f}", flush=True)\n        else:\n            evaluation_status = "skipped_missing_targets"\n            print(f"정답 포함 JSON {labeled_count:,}/{len(paths):,}개: 평가는 생략하고 예측 CSV를 저장했습니다.", flush=True)\n\n    summary = {\n        "model": "mission3_ver4", "checkpoint_path": str(checkpoint_path),\n        "model_signature": manifest.get("signature"), "folds": manifest["folds"],\n        "best_epochs_from_training_records": best_epochs,\n        "calibrator": calibrator, "retrained": False, "recalibrated": False,\n        "label_dir": str(label_dir), "samples": len(paths), "device": device,\n        "inference_batch_size": config["batch_size"],\n        "window_batch_size": manifest["config"]["window_batch_size"],\n        "versions": versions, "started_at_utc": started_at,\n        "inference_seconds": inference_seconds, "total_seconds": time.monotonic() - started,\n        "prediction_path": str(prediction_path), "evaluation_status": evaluation_status,\n        "labeled_samples": labeled_count,\n        "metrics_path": str(output_dir / "validation_metrics.json") if metrics is not None else None,\n        "macro_f1": metrics["macro_f1"] if metrics is not None else None,\n    }\n    runtime_data.write_json(output_dir / "inference_summary.json", summary)\n    print(f"예측 소요: {inference_seconds / 60:.1f}분", flush=True)\n    print("CSV:", prediction_path, flush=True)\n\n\nif __name__ == "__main__":\n    main(sys.argv[1])\n'}
runtime_dir = Path(tempfile.mkdtemp(prefix="mission3_ver4_inference_"))
for name, source in RUNTIME_SOURCES.items():
    (runtime_dir / name).write_text(source, encoding="utf-8")
runtime_config_path = runtime_dir / "runtime_config.json"
runtime_config_path.write_text(json.dumps({
    "checkpoint_path": str(checkpoint_path), "label_dir": str(LABEL_DIR),
    "output_dir": str(OUTPUT_DIR), "device": DEVICE,
    "batch_size": INFERENCE_BATCH_SIZE, "run_evaluation": RUN_EVALUATION,
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("추론 코드 준비 완료")


## 5. 저장된 모델로 예측 및 평가

`Predict: fold_1.ckpt`부터 `fold_5.ckpt`까지 나오면 **학습이 아니라 저장된 모델의 추론**입니다.
데이터 크기, GPU와 Drive 읽기 속도에 따라 시간이 달라집니다. 배치 크기와 GPU에 따라
부동소수점 계산에 작은 차이가 생길 수 있으므로 기존 점수의 완전한 일치를 보장하지는 않습니다.
기존 전체 Validation에서 기록된 Macro F1은 **0.656649**입니다.

CSV 형식은 `label file name,symptom`이며, 증상이 없으면 `[]`로 저장합니다.


In [ ]:
# 새 프로세스로 실행해 설치 전 커널의 패키지/모듈 캐시와 분리합니다.
command = [sys.executable, "-u", str(runtime_dir / "runner.py"), str(runtime_config_path)]
process = subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, encoding="utf-8", errors="replace",
)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
except BaseException:
    # 노트북에서 실행 중지 시 추론 프로세스도 중지합니다.
    process.kill()
    process.wait()
    raise
finally:
    process.stdout.close()
if return_code:
    raise subprocess.CalledProcessError(return_code, command)


In [ ]:
# 6. 결과 확인 — 기존 학습 결과는 유지됩니다.
import csv
from itertools import islice

summary_path = OUTPUT_DIR / "inference_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
print("완료:", summary["samples"], "개")
print("제출 CSV:", summary["prediction_path"])
print("실행 기록:", summary_path)
if summary["metrics_path"]:
    print("평가 결과:", summary["metrics_path"])
    print(f"Macro F1: {summary['macro_f1']:.6f}")
else:
    print("평가:", summary["evaluation_status"])
with prediction_path.open(encoding="utf-8-sig", newline="") as handle:
    for row in islice(csv.reader(handle), 6):
        print(row)


예측 결과는 **`inference_outputs/실행시각/mission3.csv`**, 평가 결과는 같은 폴더의
`validation_metrics.json`, 적용한 모델·보정값·실행 시간은 `inference_summary.json`에 저장됩니다.
기본 설정에서는 Drive에 남으므로 런타임이 종료되어도 결과를 다시 확인할 수 있습니다.

다른 JSON을 예측하려면 첫 설정 셀의 `LABEL_DIR`을 바꾸고 `OUTPUT_DIR = None`으로 두어
다시 모두 실행하세요. 학습 모델과 보정값을 다시 만들 필요가 없습니다.
